# 🛒 Amazon E-Commerce Data Analysis
### SQL-Based Business Intelligence Notebook

**Dataset Tables:**
- `orders` — 2000 transactions with pricing, discounts, tax, shipping
- `customers` — 500 customers with city, state, segment, loyalty points
- `products` — 144 products with category, brand, cost, sale price, rating
- `returns` — 167 return records with reasons and refund amounts

---
## ⚙️ Setup: Load Data into SQL Database

In [1]:
# Install required libraries
!pip install pandas openpyxl sqlalchemy --quiet


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
from sqlalchemy import create_engine, text

# ── Load all sheets from Excel ──
FILE_PATH = "G:\ecommerce_dataset_v3.xlsx"   # update path if needed

orders    = pd.read_excel(FILE_PATH, sheet_name="Orders")
customers = pd.read_excel(FILE_PATH, sheet_name="Customers")
products  = pd.read_excel(FILE_PATH, sheet_name="Products")
returns   = pd.read_excel(FILE_PATH, sheet_name="Returns")

# ── Create in-memory SQLite database via SQLAlchemy ──
engine = create_engine("sqlite:///:memory:", echo=False)

orders.to_sql("orders",    engine, index=False, if_exists="replace")
customers.to_sql("customers", engine, index=False, if_exists="replace")
products.to_sql("products",  engine, index=False, if_exists="replace")
returns.to_sql("returns",   engine, index=False, if_exists="replace")

print("✅ All tables loaded successfully!")
print(f"   orders    → {len(orders):,} rows")
print(f"   customers → {len(customers):,} rows")
print(f"   products  → {len(products):,} rows")
print(f"   returns   → {len(returns):,} rows")

<>:5: SyntaxWarning: invalid escape sequence '\e'
<>:5: SyntaxWarning: invalid escape sequence '\e'
C:\Users\harsh\AppData\Local\Temp\ipykernel_30228\1449013415.py:5: SyntaxWarning: invalid escape sequence '\e'
  FILE_PATH = "G:\ecommerce_dataset_v3.xlsx"   # update path if needed


✅ All tables loaded successfully!
   orders    → 2,000 rows
   customers → 500 rows
   products  → 144 rows
   returns   → 167 rows


In [4]:
# Helper function to run SQL and display results
def run_query(sql):
    with engine.connect() as conn:
        df = pd.read_sql(text(sql), conn)
    return df

---
## 📊 Query 1 — Total Revenue, Orders & Avg Order Value by Year

**Business Question:** How has the business grown year over year?

This query tells us the **total revenue earned**, **number of orders placed**, and the **average order value** for each year. It's one of the most important metrics to track business health and growth momentum.

In [5]:
q1 = """
SELECT
    OrderYear                                          AS Year,
    COUNT(OrderID)                                     AS Total_Orders,
    ROUND(SUM(TotalAmount), 2)                         AS Total_Revenue,
    ROUND(AVG(TotalAmount), 2)                         AS Avg_Order_Value,
    ROUND(SUM(TotalAmount) - SUM(UnitCost * Quantity), 2) AS Gross_Profit
FROM orders
GROUP BY OrderYear
ORDER BY OrderYear
"""
run_query(q1)

,Year,Total_Orders,Total_Revenue,Avg_Order_Value,Gross_Profit
0,2022,435,19871790.95,45682.28,9305775.84
1,2023,665,29610109.89,44526.48,14126339.68
2,2024,900,41020758.99,45578.62,19329338.03


> 💡 **Insight:** Compare revenue and gross profit year-over-year — if revenue grows but profit lags, rising costs are eroding margins. Declining order counts signal an acquisition problem; declining AOV signals an upsell problem.

---

---
## 📦 Query 2 — Top 10 Best-Selling Product Categories by Revenue

**Business Question:** Which product categories are driving the most sales?

Knowing which categories generate the most revenue helps the business decide **where to invest more inventory**, **run targeted promotions**, and **focus marketing spend**. Categories with high revenue but low profit margins may need pricing adjustments.

In [6]:
q2 = """
SELECT
    p.Category,
    COUNT(o.OrderID)               AS Total_Orders,
    SUM(o.Quantity)                AS Units_Sold,
    ROUND(SUM(o.TotalAmount), 2)   AS Total_Revenue,
    ROUND(AVG(o.UnitPrice), 2)     AS Avg_Unit_Price
FROM orders o
JOIN products p ON o.ProductID = p.ProductID
GROUP BY p.Category
ORDER BY Total_Revenue DESC
LIMIT 10
"""
run_query(q2)

,Category,Total_Orders,Units_Sold,Total_Revenue,Avg_Unit_Price
0,Beauty,341,1081,17091449.83,14837.53
1,Sports,342,1045,16986330.45,16129.38
2,Books,349,1041,15927701.09,14908.03
3,Clothing,347,1049,14755271.93,14010.87
4,Electronics,320,941,13394322.76,13942.27
5,Home & Kitchen,301,897,12347583.77,13451.52


> 💡 **Insight:** Top categories likely drive 70–80% of revenue — prioritize them for inventory and ad spend. Identify high-order-count categories with low revenue as potential underpricing opportunities.

---

---
## 🏆 Query 3 — Top 10 Customers by Total Spend

**Business Question:** Who are our most valuable customers?

The top 10 customers often contribute a disproportionately large share of revenue (Pareto Principle). Identifying them allows the business to offer **VIP treatment, exclusive deals, loyalty rewards**, and dedicated account management to retain them.

In [7]:
q3 = """
SELECT
    c.CustomerID,
    c.CustomerName,
    c.City,
    c.CustomerSegment,
    COUNT(o.OrderID)             AS Total_Orders,
    ROUND(SUM(o.TotalAmount), 2) AS Total_Spend,
    c.LoyaltyPoints
FROM orders o
JOIN customers c ON o.CustomerID = c.CustomerID
GROUP BY c.CustomerID, c.CustomerName, c.City, c.CustomerSegment, c.LoyaltyPoints
ORDER BY Total_Spend DESC
LIMIT 10
"""
run_query(q3)

,CustomerID,CustomerName,City,CustomerSegment,Total_Orders,Total_Spend,LoyaltyPoints
0,CUST-0220,Deepak Shah,Surat,Home Office,9,583536.96,3108
1,CUST-0005,Vivek Bose,Surat,Corporate,8,566762.96,4426
2,CUST-0017,Amit Joshi,Surat,Corporate,4,547737.69,4117
3,CUST-0338,Priya Reddy,Mumbai,Corporate,12,525418.53,4929
4,CUST-0466,Divya Patel,Jaipur,Corporate,7,523667.38,2504
5,CUST-0473,Amit Rao,Bangalore,Consumer,9,516591.44,1687
6,CUST-0189,Pooja Rao,Delhi,Consumer,7,497013.40,1147
7,CUST-0096,Pooja Iyer,Pune,Consumer,5,493849.45,4875
8,CUST-0041,Amit Bose,Delhi,Consumer,6,468603.55,2363
9,CUST-0337,Rekha Iyer,Pune,Corporate,7,461680.88,1166


> 💡 **Insight:** High-spend customers with low loyalty points are likely unenrolled in the rewards program — a missed retention opportunity. Cross-reference segment to tailor offers (e.g., credit terms for Corporate buyers).

---

---
## 💳 Query 4 — Revenue & Orders by Payment Method

**Business Question:** Which payment methods are customers preferring?

Understanding payment preferences helps the business **optimize checkout flows**, **negotiate better rates with payment partners**, and **identify if certain payment methods have higher order values**. For example, EMI-based purchases often lead to higher basket sizes.

In [8]:
q4 = """
SELECT
    PaymentMethod,
    COUNT(OrderID)                   AS Total_Orders,
    ROUND(SUM(TotalAmount), 2)       AS Total_Revenue,
    ROUND(AVG(TotalAmount), 2)       AS Avg_Order_Value,
    ROUND(100.0 * COUNT(OrderID) / (SELECT COUNT(*) FROM orders), 2) AS Order_Share_Pct
FROM orders
GROUP BY PaymentMethod
ORDER BY Total_Revenue DESC
"""
run_query(q4)

,PaymentMethod,Total_Orders,Total_Revenue,Avg_Order_Value,Order_Share_Pct
0,Credit Card,341,16665472.32,48872.35,17.05
1,Cash on Delivery,365,16614080.18,45518.03,18.25
2,Net Banking,361,15425619.46,42730.25,18.05
3,EMI,309,14538917.23,47051.51,15.45
4,UPI,313,13760942.13,43964.67,15.65
5,Debit Card,311,13497628.51,43400.73,15.55


> 💡 **Insight:** Payment methods with higher AOV (e.g., EMI, credit card) should be prominently featured at checkout. A dominant UPI/wallet share signals mobile-first behavior — ensure a frictionless flow for those methods.

---

---
## 📱 Query 5 — Sales Channel Performance (Website vs App vs Other)

**Business Question:** Which sales channel brings in the most revenue?

This shows the **performance of each sales channel** — Mobile App, Website, etc. A growing mobile app channel signals that investing in **app features, push notifications, and app-exclusive offers** can drive significant revenue uplift.

In [9]:
q5 = """
SELECT
    SalesChannel,
    COUNT(OrderID)                   AS Total_Orders,
    ROUND(SUM(TotalAmount), 2)       AS Total_Revenue,
    ROUND(AVG(TotalAmount), 2)       AS Avg_Order_Value,
    ROUND(AVG(DiscountRate) * 100, 2) AS Avg_Discount_Pct
FROM orders
GROUP BY SalesChannel
ORDER BY Total_Revenue DESC
"""
run_query(q5)

,SalesChannel,Total_Orders,Total_Revenue,Avg_Order_Value,Avg_Discount_Pct
0,Website,670,30322924.47,45258.10,12.47
1,Third Party,681,30140101.50,44258.59,12.33
2,Mobile App,649,30039633.86,46286.03,12.37


> 💡 **Insight:** If the app shows higher average discounts than the website, app-exclusive offers may be compressing margins. Invest in the highest-revenue channel and track conversion rate alongside revenue.

---

---
## 📅 Query 6 — Monthly Revenue Trend (All Years)

**Business Question:** Are there seasonal spikes or dips in sales throughout the year?

Seasonality analysis helps the business **plan inventory ahead of peak months** (e.g., festival seasons, year-end sales), **hire temporary staff**, and **run promotions during slow months** to smoothen revenue. Months like October–December typically see higher e-commerce sales.

In [10]:
q6 = """
SELECT
    OrderYear                        AS Year,
    OrderMonth                       AS Month,
    COUNT(OrderID)                   AS Total_Orders,
    ROUND(SUM(TotalAmount), 2)       AS Monthly_Revenue,
    ROUND(SUM(DiscountAmount), 2)    AS Total_Discounts_Given
FROM orders
GROUP BY OrderYear, OrderMonth
ORDER BY OrderYear, OrderMonth
"""
run_query(q6)

,Year,Month,Total_Orders,Monthly_Revenue,Total_Discounts_Given
0,2022,1,33,1525924.69,224104.38
1,2022,2,21,951569.18,91172.98
2,2022,3,34,1996090.58,253998.44
3,2022,4,30,1314114.87,149948.40
4,2022,5,39,1603127.65,203072.96
5,2022,6,41,2006378.65,237098.13
6,2022,7,42,1739002.85,174199.21
7,2022,8,41,1685824.50,244744.95
8,2022,9,31,1700831.09,228115.88
9,2022,10,48,2112259.36,259110.53


> 💡 **Insight:** Q4 months typically spike due to festive seasons — pre-position inventory 6–8 weeks ahead. Use consistently low-revenue months for clearance campaigns or new product launches.

---

---
## 🔄 Query 7 — Return Rate & Top Return Reasons

**Business Question:** How often are products being returned, and why?

A high return rate eats into profitability. This query helps identify the **most common return reasons** (damaged product, wrong item, late delivery), so the business can take corrective action — better packaging, faster shipping, or stricter quality control. The refund amount shows the **financial impact** of returns.

In [12]:
q7 = """
SELECT
    ReturnReason,
    ReturnStatus,
    COUNT(ReturnID)              AS Total_Returns,
    ROUND(SUM(RefundAmount), 2)  AS Total_Refund_Amount,
    ROUND(AVG(RefundAmount), 2)  AS Avg_Refund_Amount
FROM returns 
GROUP BY ReturnReason,ReturnStatus
ORDER BY Total_Returns DESC
"""
run_query(q7)

,ReturnReason,ReturnStatus,Total_Returns,Total_Refund_Amount,Avg_Refund_Amount
0,Wrong Item,Approved,27,890654.23,32987.19
1,Damaged Product,Approved,21,744569.12,35455.67
2,Size Issue,Approved,21,989880.46,47137.16
3,Changed Mind,Approved,20,732691.26,36634.56
4,Quality Issue,Approved,19,468098.79,24636.78
5,Late Delivery,Approved,15,782858.43,52190.56
6,Late Delivery,Rejected,8,322691.89,40336.49
7,Wrong Item,Pending,7,226311.70,32330.24
8,Quality Issue,Pending,6,204656.10,34109.35
9,Size Issue,Pending,6,201433.74,33572.29


> 💡 **Insight:** Top return reasons directly expose operational failures — damaged items point to packaging issues, wrong items to dispatch errors, late delivery to courier SLA gaps. Quantify the refund cost per reason to prioritize fixes.

---

---
## 🌆 Query 8 — Top 10 Cities by Revenue

**Business Question:** Which cities generate the most revenue for us?

Geo-analysis helps the business **prioritize delivery infrastructure, warehousing, and hyperlocal marketing** in high-revenue cities. It also reveals untapped markets in cities with high population but low order counts — potential areas for growth campaigns.

In [13]:
q8 = """
SELECT
    c.City,
    c.State,
    COUNT(o.OrderID)             AS Total_Orders,
    COUNT(DISTINCT o.CustomerID) AS Unique_Customers,
    ROUND(SUM(o.TotalAmount), 2) AS Total_Revenue
FROM orders o
JOIN customers c ON o.CustomerID = c.CustomerID
GROUP BY c.City, c.State
ORDER BY Total_Revenue DESC
LIMIT 10
"""
run_query(q8)

,City,State,Total_Orders,Unique_Customers,Total_Revenue
0,Kolkata,West Bengal,228,63,11184762.77
1,Delhi,Delhi,209,54,10130184.87
2,Surat,Gujarat,216,50,10033116.94
3,Jaipur,Rajasthan,219,51,9398885.23
4,Pune,Maharashtra,200,49,9288342.72
5,Bangalore,Karnataka,197,49,9225889.39
6,Ahmedabad,Gujarat,204,49,8116442.78
7,Chennai,Tamil Nadu,184,48,8024405.38
8,Mumbai,Maharashtra,177,45,7819433.77
9,Hyderabad,Telangana,166,36,7281195.98


> 💡 **Insight:** Tier-2 cities appearing in the top 10 signal high-growth markets with lower acquisition costs — worth targeted hyper-local campaigns. A high orders-to-unique-customers ratio in a city means strong repeat behavior there.

---

---
## 👥 Query 9 — Revenue by Customer Segment

**Business Question:** Which customer segments are most profitable?

Customer segments (Corporate, Home Office, Consumer, etc.) have different buying behaviors and price sensitivities. This analysis guides **segment-specific pricing strategies, bulk discount policies, and B2B vs B2C marketing budgets**.

In [14]:
q9 = """
SELECT
    c.CustomerSegment,
    COUNT(DISTINCT o.CustomerID)    AS Unique_Customers,
    COUNT(o.OrderID)                AS Total_Orders,
    ROUND(SUM(o.TotalAmount), 2)    AS Total_Revenue,
    ROUND(AVG(o.TotalAmount), 2)    AS Avg_Order_Value,
    ROUND(AVG(o.DiscountRate)*100, 2) AS Avg_Discount_Pct
FROM orders o
JOIN customers c ON o.CustomerID = c.CustomerID
GROUP BY c.CustomerSegment
ORDER BY Total_Revenue DESC
"""
run_query(q9)

,CustomerSegment,Unique_Customers,Total_Orders,Total_Revenue,Avg_Order_Value,Avg_Discount_Pct
0,Consumer,185,724,32691025.27,45153.35,12.25
1,Corporate,153,609,28924450.22,47494.99,12.63
2,Home Office,156,667,28887184.34,43309.12,12.33


> 💡 **Insight:** High discount rates in a segment may be compressing margins without driving incremental volume — evaluate whether discounts are truly necessary there. Design segment-specific offers instead of blanket platform-wide discounts.

---

---
## ⭐ Query 10 — Top 10 Products by Rating & Sales Volume

**Business Question:** Which products are both highly rated AND selling well?

Products with **high ratings AND high sales volume** are your star performers — they should be featured prominently in search results, ads, and homepage banners. Products with high ratings but low sales may just need better visibility or pricing adjustments.

In [15]:
q10 = """
SELECT
    p.ProductID,
    p.Category,
    p.SubCategory,
    p.Brand,
    p.Rating,
    COUNT(o.OrderID)             AS Total_Orders,
    SUM(o.Quantity)              AS Units_Sold,
    ROUND(SUM(o.TotalAmount), 2) AS Total_Revenue
FROM products p
JOIN orders o ON p.ProductID = o.ProductID
GROUP BY p.ProductID, p.Category, p.SubCategory, p.Brand, p.Rating
ORDER BY p.Rating DESC, Units_Sold DESC
LIMIT 10
"""
run_query(q10)

,ProductID,Category,SubCategory,Brand,Rating,Total_Orders,Units_Sold,Total_Revenue
0,BTY-SUN-003,Beauty,Sunscreen,Samsung,5.0,12,40,941288.07
1,ELE-SPK-001,Electronics,Speaker,Apple,5.0,4,9,49289.24
2,BTY-NPL-003,Beauty,Nail Polish,Samsung,4.9,19,70,1316361.83
3,HOM-VSE-001,Home & Kitchen,Vase,Lakme,4.9,20,59,469858.53
4,HOM-PCK-001,Home & Kitchen,Pressure Cooker,Boat,4.9,17,55,794275.69
5,CLO-JKT-001,Clothing,Jacket,Sony,4.9,16,47,618430.12
6,CLO-SHO-001,Clothing,Shoes,Sony,4.9,13,45,586795.09
7,SPT-DMB-003,Sports,Dumbbell,Lakme,4.9,13,44,1704934.40
8,CLO-KRT-001,Clothing,Kurta,Adidas,4.9,12,40,307947.21
9,HOM-BDS-003,Home & Kitchen,Bed Sheet,Prestige,4.9,10,38,533714.59


> 💡 **Insight:** Products with high ratings but low sales are hidden gems — they just need better visibility (featured placement, sponsored listings). Use star products as anchor items in cross-sell bundles and protect them from stockouts.

---

---
## 💰 Query 11 — Profit Margin Analysis by Product Category

**Business Question:** Which categories are most and least profitable?

Revenue alone is not enough — a category could have high revenue but thin margins. This query shows **gross profit and margin percentage** per category, helping the business decide where to **cut costs, renegotiate supplier prices**, or **increase selling prices**.

In [16]:
q11 = """
SELECT
    p.Category,
    ROUND(SUM(o.TotalAmount), 2)                           AS Total_Revenue,
    ROUND(SUM(o.UnitCost * o.Quantity), 2)                 AS Total_Cost,
    ROUND(SUM(o.TotalAmount) - SUM(o.UnitCost * o.Quantity), 2)  AS Gross_Profit,
    ROUND(
        100.0 * (SUM(o.TotalAmount) - SUM(o.UnitCost * o.Quantity))
        / NULLIF(SUM(o.TotalAmount), 0), 2
    ) AS Profit_Margin_Pct
FROM orders o
JOIN products p ON o.ProductID = p.ProductID
GROUP BY p.Category
ORDER BY Profit_Margin_Pct DESC
"""
run_query(q11)

,Category,Total_Revenue,Total_Cost,Gross_Profit,Profit_Margin_Pct
0,Sports,16986330.45,8687612.28,8298718.17,48.86
1,Beauty,17091449.83,8773755.97,8317693.86,48.67
2,Home & Kitchen,12347583.77,6512692.37,5834891.40,47.26
3,Electronics,13394322.76,7113435.93,6280886.83,46.89
4,Clothing,14755271.93,7842190.32,6913081.61,46.85
5,Books,15927701.09,8811519.41,7116181.68,44.68


> 💡 **Insight:** High-revenue but low-margin categories are vulnerable — even a small cost increase can turn them loss-making. Avoid deep discounting in already thin-margin categories and consider renegotiating supplier contracts there first.

---

---
## 🚚 Query 12 — Order Status Distribution & Shipping Cost Analysis

**Business Question:** What percentage of orders are delivered, pending, or cancelled?

A high percentage of pending or cancelled orders signals **operational issues** — poor logistics, payment failures, or stock-outs. Monitoring shipping costs helps identify if **logistics expenses are eating into margins** and whether courier negotiations are needed.

In [17]:
q12 = """
SELECT
    OrderStatus,
    COUNT(OrderID)                                           AS Total_Orders,
    ROUND(100.0 * COUNT(OrderID) / (SELECT COUNT(*) FROM orders), 2) AS Share_Pct,
    ROUND(SUM(TotalAmount), 2)                               AS Total_Revenue,
    ROUND(AVG(ShippingCost), 2)                              AS Avg_Shipping_Cost,
    ROUND(SUM(ShippingCost), 2)                              AS Total_Shipping_Cost
FROM orders
GROUP BY OrderStatus
ORDER BY Total_Orders DESC
"""
run_query(q12)

,OrderStatus,Total_Orders,Share_Pct,Total_Revenue,Avg_Shipping_Cost,Total_Shipping_Cost
0,Delivered,1395,69.75,62934223.83,76.21,106308.51
1,Shipped,237,11.85,10252159.89,75.36,17860.39
2,Cancelled,213,10.65,9807253.57,80.19,17080.54
3,Processing,155,7.75,7509022.54,72.19,11188.83


> 💡 **Insight:** A cancellation rate above 5–8% represents wasted fulfillment cost and customer churn risk. If shipping cost is growing faster than order volume, it's time to renegotiate courier contracts.

---

---
## 🎯 Query 13 — Discount Impact: High vs Low Discount Orders

**Business Question:** Are heavy discounts actually driving more revenue, or are they hurting profitability?

This query buckets orders by discount level and compares average order value and profit. If **high-discount orders generate lower total revenue**, it indicates that discounts are not being used strategically and the business could be leaking margin unnecessarily.

In [19]:
q13 = """
SELECT
    CASE
        WHEN DiscountRate = 0           THEN '0% — No Discount'
        WHEN DiscountRate <= 0.10       THEN '1–10% — Low'
        WHEN DiscountRate <= 0.20       THEN '11–20% — Medium'
        WHEN DiscountRate <= 0.30       THEN '21–30% — High'
        ELSE '31%+ — Very High'
    END AS Discount_Bucket,
    COUNT(OrderID) AS Total_Orders,
    ROUND(AVG(TotalAmount), 2) AS Avg_Order_Value,
    ROUND(SUM(TotalAmount), 2) AS Total_Revenue,
    ROUND(SUM(DiscountAmount), 2) AS Total_Discount_Given
FROM orders
GROUP BY Discount_Bucket
ORDER BY Total_Revenue DESC
"""
run_query(q13)

,Discount_Bucket,Total_Orders,Avg_Order_Value,Total_Revenue,Total_Discount_Given
0,1–10% — Low,833,48642.14,40518904.83,1929602.65
1,11–20% — Medium,773,43146.25,33352049.01,5225729.11
2,21–30% — High,359,41208.63,14793896.89,3692256.72
3,0% — No Discount,35,52508.83,1837809.10,0.00


> 💡 **Insight:** If no-discount orders have a high AOV, many customers buy at full price — blanket discounting is pure margin leakage on them. Shift to behavioral discounts triggered only by cart abandonment or price-sensitivity signals.

---

---
## 📉 Query 14 — Products with Highest Return Rate

**Business Question:** Which specific products are being returned the most?

Products with a high return rate signal **quality issues, misleading descriptions, or sizing problems**. This analysis enables the business to take direct action — improving product pages, contacting suppliers, or temporarily pulling a product from listings until the issue is resolved.

In [20]:
q14 = """
SELECT
    o.ProductID,
    p.Category,
    p.SubCategory,
    COUNT(DISTINCT o.OrderID)          AS Total_Orders,
    COUNT(DISTINCT r.ReturnID)         AS Total_Returns,
    ROUND(
        100.0 * COUNT(DISTINCT r.ReturnID) / NULLIF(COUNT(DISTINCT o.OrderID), 0)
    , 2)                               AS Return_Rate_Pct,
    ROUND(SUM(r.RefundAmount), 2)      AS Total_Refunded
FROM orders o
JOIN products p ON o.ProductID = p.ProductID
LEFT JOIN returns r ON o.OrderID = r.OrderID
GROUP BY o.ProductID, p.Category, p.SubCategory
HAVING Total_Returns > 0
ORDER BY Return_Rate_Pct DESC
LIMIT 10
"""
run_query(q14)

,ProductID,Category,SubCategory,Total_Orders,Total_Returns,Return_Rate_Pct,Total_Refunded
0,HOM-PIL-001,Home & Kitchen,Pillow,11,3,27.27,103325.27
1,ELE-HPH-002,Electronics,Headphones,15,4,26.67,48364.62
2,BOK-COK-002,Books,Cooking,12,3,25.00,276163.14
3,BTY-FWS-002,Beauty,Face Wash,16,4,25.00,148023.12
4,BTY-SUN-003,Beauty,Sunscreen,12,3,25.00,149092.25
5,ELE-SPK-001,Electronics,Speaker,4,1,25.00,3316.36
6,ELE-SPK-002,Electronics,Speaker,12,3,25.00,112472.42
7,HOM-CRT-003,Home & Kitchen,Curtain,12,3,25.00,70620.07
8,HOM-TST-002,Home & Kitchen,Toaster,8,2,25.00,13805.87
9,BOK-TCH-002,Books,Technology,13,3,23.08,145357.15


> 💡 **Insight:** A return rate above 15–20% on a product destroys CLV and inflates reverse-logistics costs — audit listings for misleading descriptions and engage the supplier. Products from the same sub-category appearing repeatedly suggest a systemic supplier quality issue.

---

---
## 🔁 Query 15 — Repeat Customer Analysis (Single vs Multi-Order Customers)

**Business Question:** How many customers have placed more than one order? What is their revenue contribution?

Repeat customers are the backbone of sustainable e-commerce. **Acquiring a new customer costs 5–7x more than retaining an existing one.** This query shows what proportion of revenue comes from loyal repeat buyers vs one-time buyers, directly informing **retention and loyalty program strategy**.

In [21]:
q15 = """
WITH customer_orders AS (
    SELECT
        CustomerID,
        COUNT(OrderID)             AS Order_Count,
        ROUND(SUM(TotalAmount), 2) AS Total_Spend
    FROM orders
    GROUP BY CustomerID
)
SELECT
    CASE
        WHEN Order_Count = 1 THEN 'One-Time Buyer'
        WHEN Order_Count BETWEEN 2 AND 4 THEN 'Occasional Buyer (2–4 orders)'
        ELSE 'Loyal Buyer (5+ orders)'
    END                            AS Customer_Type,
    COUNT(CustomerID)              AS Num_Customers,
    ROUND(SUM(Total_Spend), 2)     AS Total_Revenue,
    ROUND(AVG(Total_Spend), 2)     AS Avg_Spend_Per_Customer,
    ROUND(AVG(Order_Count), 2)     AS Avg_Orders_Per_Customer
FROM customer_orders
GROUP BY Customer_Type
ORDER BY Total_Revenue DESC
"""
run_query(q15)

,Customer_Type,Num_Customers,Total_Revenue,Avg_Spend_Per_Customer,Avg_Orders_Per_Customer
0,Loyal Buyer (5+ orders),189,49831031.84,263656.25,6.08
1,Occasional Buyer (2–4 orders),272,39295542.92,144468.91,3.01
2,One-Time Buyer,33,1376085.07,41699.55,1.00


> 💡 **Insight:** If one-time buyers dominate, the business is good at acquisition but poor at retention — an expensive treadmill. A structured post-purchase email sequence (Day 7, 30, 60) can meaningfully lift one-time-to-repeat conversion.

---

---
## ✅ Summary of Key Business Insights

| # | Query | Key Business Use |
|---|-------|------------------|
| 1 | YoY Revenue & Profit | Track business growth |
| 2 | Top Categories by Revenue | Focus inventory & marketing |
| 3 | Top 10 Customers | VIP retention strategies |
| 4 | Payment Method Analysis | Optimize checkout & partnerships |
| 5 | Sales Channel Performance | Invest in best-performing channel |
| 6 | Monthly Revenue Trend | Seasonal planning & promotions |
| 7 | Return Reasons & Refunds | Fix quality & logistics issues |
| 8 | Top Cities by Revenue | Geo-targeted marketing & warehousing |
| 9 | Customer Segment Analysis | Segment-specific pricing & offers |
| 10 | High-Rating + High-Sales Products | Feature star products in ads |
| 11 | Profit Margin by Category | Identify low-margin categories |
| 12 | Order Status & Shipping Cost | Monitor ops & logistics costs |
| 13 | Discount Impact Analysis | Avoid margin leakage from over-discounting |
| 14 | High Return-Rate Products | Fix or delist problem products |
| 15 | Repeat vs One-Time Buyers | Build loyalty & retention programs |

> 💡 **Tip:** Use these queries as a foundation. Connect to a live database (MySQL / PostgreSQL) by replacing `sqlite:///:memory:` with your actual connection string in SQLAlchemy.